# Ddri 지구판단용 군집화 피처 생성 노트북

목적:
- 2차 군집화에서 `지구판단`에 직접적으로 쓰기 위한 도착 시간대 중심 피처를 생성한다.
- 기존 운영 성격 피처와 섞지 않고, 별도 CSV로 생성해 팀 검토용 산출물로 관리한다.
- baseline 전처리 함수를 그대로 재사용해 표본 기준을 맞춘다.

이번 노트북에서 만드는 핵심 피처:
- `arrival_7_10_ratio` : 7~10시 도착 비율
- `arrival_11_14_ratio` : 11~14시 도착 비율
- `arrival_17_20_ratio` : 17~20시 도착 비율
- `morning_net_inflow` : 7~10시 순유입
- `evening_net_inflow` : 17~20시 순유입

## 왜 도착 기준 피처를 먼저 보는가

지구판단을 하려면 `어디서 출발했는가`보다 `어디로 도착했는가`가 더 직접적일 수 있다.

예를 들면:
- 아침 시간대 도착이 많으면 업무지구 / 상업업무 혼합지구 가능성
- 점심 시간대 도착이 많으면 오피스권 / 상권 중심지 가능성
- 저녁 시간대 도착이 많으면 주거지구 가능성

그래서 이 노트북은 출발 기준 운영 피처보다, 도착 시간대 중심 피처를 우선 생성한다.

## 입력과 출력

입력:
- 원천 따릉이 이용 로그 CSV
- baseline 스크립트의 공통 대여소 / 유효 반납 대여소 기준

출력:
- `cheng80/ddri_station_cluster_district_features_train_2023_2024.csv`
- `cheng80/ddri_station_cluster_district_features_test_2025.csv`

주의:
- 기존 군집 feature CSV와 바로 병합하지 않는다.
- 발표용 / 검토용 별도 산출물로 관리한다.

## 전처리 기준

이 노트북은 `works/01_clustering/01_baseline/ddri_station_clustering_baseline.py`의 함수를 그대로 불러와 사용한다.

즉 다음 기준이 동일하게 적용된다.
- 필수 컬럼 결측 제거
- `이용시간(분) <= 0` 제거
- `이용거리(M) <= 0` 제거
- `동일 대여소 반납 + 5분 이하` 제거
- 공통 대여소 기준 밖 대여 제거
- 해당 연도 강남구 기준 밖 반납 제거

이 기준을 baseline과 맞춰야, 이후 군집화 feature와 비교할 때 해석이 흔들리지 않는다.

In [1]:
from pathlib import Path
import sys

import pandas as pd

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work')
BASELINE_DIR = BASE_DIR / 'works' / '01_clustering' / '01_baseline'
OUTPUT_DATA_DIR = BASE_DIR / 'cheng80'

if str(BASELINE_DIR) not in sys.path:
    sys.path.append(str(BASELINE_DIR))

from ddri_station_clustering_baseline import file_groups, load_clean_group, load_station_frames

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
BASE_DIR, OUTPUT_DATA_DIR

(PosixPath('/Users/cheng80/Desktop/ddri_work'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/cheng80'))

## 도착 시간대 비율 피처 함수

도착 기준 시간대 비율은 `반납일시.hour`를 사용한다.

시간대 정의:
- 아침 도착: `07~10시`
- 점심 도착: `11~14시`
- 저녁 도착: `17~20시`

각 비율은 `해당 시간대 반납 수 / 전체 반납 수`로 계산한다.

In [2]:
def build_arrival_ratio_features(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work['return_hour'] = work['반납일시'].dt.hour

    total_returns = work.groupby('반납대여소번호').size()
    arrival_7_10 = work.loc[work['return_hour'].between(7, 10)].groupby('반납대여소번호').size()
    arrival_11_14 = work.loc[work['return_hour'].between(11, 14)].groupby('반납대여소번호').size()
    arrival_17_20 = work.loc[work['return_hour'].between(17, 20)].groupby('반납대여소번호').size()

    ratio_df = pd.DataFrame({'station_id': total_returns.index})
    ratio_df['arrival_7_10_ratio'] = (arrival_7_10 / total_returns).reindex(total_returns.index).fillna(0.0).values
    ratio_df['arrival_11_14_ratio'] = (arrival_11_14 / total_returns).reindex(total_returns.index).fillna(0.0).values
    ratio_df['arrival_17_20_ratio'] = (arrival_17_20 / total_returns).reindex(total_returns.index).fillna(0.0).values
    return ratio_df.sort_values('station_id').reset_index(drop=True)


## 시간대별 순유입 피처 함수

순유입은 `반납량 - 대여량`으로 정의한다.

해석:
- 양수: 그 시간대에 자전거가 들어오는 대여소
- 음수: 그 시간대에 자전거가 빠져나가는 대여소

이번 노트북에서는 지구판단에 중요한 두 구간만 우선 생성한다.
- `morning_net_inflow` : `07~10시`
- `evening_net_inflow` : `17~20시`

In [3]:
def build_time_window_net_inflow_features(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work['rental_hour'] = work['대여일시'].dt.hour
    work['return_hour'] = work['반납일시'].dt.hour

    morning_rental = work.loc[work['rental_hour'].between(7, 10)].groupby('대여 대여소번호').size()
    morning_return = work.loc[work['return_hour'].between(7, 10)].groupby('반납대여소번호').size()
    evening_rental = work.loc[work['rental_hour'].between(17, 20)].groupby('대여 대여소번호').size()
    evening_return = work.loc[work['return_hour'].between(17, 20)].groupby('반납대여소번호').size()

    station_ids = sorted(set(work['대여 대여소번호']).union(set(work['반납대여소번호'])))
    station_index = pd.Index(station_ids)

    net_df = pd.DataFrame({'station_id': station_index})
    net_df['morning_net_inflow'] = (
        morning_return.reindex(station_index).fillna(0).values
        - morning_rental.reindex(station_index).fillna(0).values
    )
    net_df['evening_net_inflow'] = (
        evening_return.reindex(station_index).fillna(0).values
        - evening_rental.reindex(station_index).fillna(0).values
    )
    return net_df.sort_values('station_id').reset_index(drop=True)


## 최종 지구판단 피처 테이블 생성 함수

도착 비율 피처와 시간대별 순유입 피처를 `station_id` 기준으로 합친다.
이번 단계에서는 최소 발표용 세트만 생성한다.

In [4]:
def build_district_features(df: pd.DataFrame) -> pd.DataFrame:
    arrival_ratio_df = build_arrival_ratio_features(df)
    net_inflow_df = build_time_window_net_inflow_features(df)

    feature_df = (
        arrival_ratio_df.merge(net_inflow_df, on='station_id', how='inner')
        .sort_values('station_id')
        .reset_index(drop=True)
    )
    return feature_df[
        [
            'station_id',
            'arrival_7_10_ratio',
            'arrival_11_14_ratio',
            'arrival_17_20_ratio',
            'morning_net_inflow',
            'evening_net_inflow',
        ]
    ]


## 데이터 로드 및 정제

학습용은 `2023 + 2024`, 테스트용은 `2025`로 분리한다.
로딩과 cleaning은 baseline 함수에 그대로 맡긴다.

In [5]:
station_id_sets, common_station_ids = load_station_frames()
common_station_id_set = set(common_station_ids)
rental_groups = file_groups()

train_df_2023, _ = load_clean_group(
    rental_groups[2023], station_id_sets[2023], common_station_id_set, 'train_2023'
)
train_df_2024, _ = load_clean_group(
    rental_groups[2024], station_id_sets[2024], common_station_id_set, 'train_2024'
)
test_df_2025, _ = load_clean_group(
    rental_groups[2025], station_id_sets[2025], common_station_id_set, 'test_2025'
)

train_df = pd.concat([train_df_2023, train_df_2024], ignore_index=True)

print('train_event_rows =', len(train_df))
print('test_event_rows =', len(test_df_2025))

[train_2023] 1/12 2301_강남구_따릉이_이용정보.csv: 34,937 -> 30,957
[train_2023] 2/12 2302_강남구_따릉이_이용정보.csv: 49,372 -> 44,188
[train_2023] 3/12 2303_강남구_따릉이_이용정보.csv: 80,572 -> 72,079


[train_2023] 4/12 2304_강남구_따릉이_이용정보.csv: 91,717 -> 81,967
[train_2023] 5/12 2305_강남구_따릉이_이용정보.csv: 107,880 -> 97,042


[train_2023] 6/12 2306_강남구_따릉이_이용정보.csv: 105,610 -> 95,109
[train_2023] 7/12 2307_강남구_따릉이_이용정보.csv: 85,691 -> 76,616


[train_2023] 8/12 2308_강남구_따릉이_이용정보.csv: 92,398 -> 82,801
[train_2023] 9/12 2309_강남구_따릉이_이용정보.csv: 105,436 -> 93,874


[train_2023] 10/12 2310_강남구_따릉이_이용정보.csv: 124,747 -> 111,821
[train_2023] 11/12 2311_강남구_따릉이_이용정보.csv: 85,518 -> 77,369
[train_2023] 12/12 2312_강남구_따릉이_이용정보.csv: 48,602 -> 43,819


...

[train_2024] 10/12 2410_강남구_따릉이_이용정보.csv: 101,558 -> 95,690
[train_2024] 11/12 2411_강남구_따릉이_이용정보.csv: 78,883 -> 73,991
[train_2024] 12/12 2412_강남구_따릉이_이용정보.csv: 48,645 -> 45,507


[test_2025] 1/12 2501_강남구_따릉이_이용정보.csv: 37,502 -> 35,410
[test_2025] 2/12 2502_강남구_따릉이_이용정보.csv: 36,065 -> 34,375
[test_2025] 3/12 2503_강남구_따릉이_이용정보.csv: 63,383 -> 61,268


[test_2025] 4/12 2504_강남구_따릉이_이용정보.csv: 85,989 -> 83,339
[test_2025] 5/12 2505_강남구_따릉이_이용정보.csv: 85,055 -> 82,574
[test_2025] 6/12 2506_강남구_따릉이_이용정보.csv: 91,073 -> 88,635


[test_2025] 7/12 2507_강남구_따릉이_이용정보.csv: 84,314 -> 79,542
[test_2025] 8/12 2508_강남구_따릉이_이용정보.csv: 86,307 -> 79,840
[test_2025] 9/12 2509_강남구_따릉이_이용정보.csv: 94,073 -> 86,051


[test_2025] 10/12 2510_강남구_따릉이_이용정보.csv: 82,459 -> 75,288
[test_2025] 11/12 2511_강남구_따릉이_이용정보.csv: 77,918 -> 71,552
[test_2025] 12/12 2512_강남구_따릉이_이용정보.csv: 45,259 -> 42,015
train_event_rows = 1844049
test_event_rows = 819889


## 지구판단 피처 생성 및 미리보기

row 수와 컬럼 구성이 예상과 맞는지 먼저 확인한다.
이 단계에서는 기존 군집 feature CSV와 병합하지 않는다.

In [6]:
train_feature_df = build_district_features(train_df)
test_feature_df = build_district_features(test_df_2025)

print('train_feature_rows =', len(train_feature_df))
print('test_feature_rows =', len(test_feature_df))

train_feature_df.head()

train_feature_rows = 174
test_feature_rows = 172


,station_id,arrival_7_10_ratio,arrival_11_14_ratio,arrival_17_20_ratio,morning_net_inflow,evening_net_inflow
0,2301,0.075694,0.126693,0.373307,7.0,1754.0
1,2302,0.128852,0.165705,0.380852,-125.0,1922.0
2,2303,0.117521,0.158425,0.359829,-2241.0,910.0
3,2304,0.161560,0.183048,0.319936,-751.0,-1352.0
4,2305,0.429047,0.202975,0.167156,1661.0,-1498.0


In [7]:
train_feature_df.describe().round(4)

,station_id,arrival_7_10_ratio,arrival_11_14_ratio,arrival_17_20_ratio,morning_net_inflow,evening_net_inflow
count,174.0000,174.0000,174.0000,174.0000,174.0000,174.0000
mean,3008.0402,0.1884,0.1786,0.2997,-88.8161,56.0115
std,956.8159,0.0936,0.0347,0.0666,2299.4478,1324.2545
min,2301.0000,0.0475,0.1018,0.1105,-8110.0000,-7627.0000
25%,2349.2500,0.1242,0.1546,0.2542,-848.0000,-516.0000
50%,2404.5000,0.1632,0.1787,0.3015,-148.5000,73.5000
75%,3627.7500,0.2408,0.2022,0.3362,384.5000,585.0000
max,4935.0000,0.5853,0.2643,0.4989,16019.0000,5340.0000


## CSV 저장

저장 파일:
- `cheng80/ddri_station_cluster_district_features_train_2023_2024.csv`
- `cheng80/ddri_station_cluster_district_features_test_2025.csv`

In [8]:
train_output_path = OUTPUT_DATA_DIR / 'ddri_station_cluster_district_features_train_2023_2024.csv'
test_output_path = OUTPUT_DATA_DIR / 'ddri_station_cluster_district_features_test_2025.csv'

train_feature_df.to_csv(train_output_path, index=False)
test_feature_df.to_csv(test_output_path, index=False)

print('saved:', train_output_path)
print('saved:', test_output_path)

saved: /Users/cheng80/Desktop/ddri_work/cheng80/ddri_station_cluster_district_features_train_2023_2024.csv
saved: /Users/cheng80/Desktop/ddri_work/cheng80/ddri_station_cluster_district_features_test_2025.csv


## 최종 메모

- 이 노트북은 `지구판단용 최소 발표 세트` 생성이 목적이다.
- 이후 교통 접근성, 고저차, POI와 결합해 확장할 수 있다.
- 팀 검토 전까지는 기존 군집화 feature와 합치지 않는다.